# 01 - Cross-entropy loss

**Goal:** understand how a language model's next-token prediction is scored, by computing cross-entropy by hand and checking it against PyTorch.

**Key ideas so far**
- The model outputs **logits**: one raw score per vocabulary token.
- **Softmax** turns logits into probabilities: `p_i = exp(logit_i) / sum_j exp(logit_j)`.
- The loss for one position looks only at the probability of the **correct** token: `loss = -log(p_correct)`.
- Over many positions, average the per-position losses.

**Tasks**
1. Create random logits of shape `(8, 65)` (8 positions, 65-char vocab) and random integer targets of shape `(8,)`.
2. Compute the loss **by hand**: softmax -> pick each correct token's probability -> `-log` -> mean.
3. Compare with `torch.nn.functional.cross_entropy(logits, targets)`. They should match.
4. **Predict first, then check:** what is the loss when all logits are zeros? (Write your prediction in a markdown cell before running it.)

**Done when:** your hand-computed loss matches PyTorch's, and the all-zeros result matches your prediction.

In [1]:
'''sample code
x = torch.randn(3, 5)                 # 3x5 tensor of random normal numbers
z = torch.zeros(3, 5)                 # 3x5 tensor of zeros
t = torch.randint(0, 5, (3,))         # 3 random ints in [0, 5), shape (3,)

x.shape                               # torch.Size([3, 5])
torch.exp(x), torch.log(x)            # elementwise
x.sum(dim=1, keepdim=True)            # sum across each row -> shape (3, 1)
x / x.sum(dim=1, keepdim=True)        # broadcasting: each row divided by its own sum
x.mean()                              # average of all elements -> a scalar tensor

# fancy indexing: pick one element per row
rows = torch.arange(3)                # tensor([0, 1, 2])
cols = torch.tensor([4, 0, 2])
x[rows, cols]                         # [x[0,4], x[1,0], x[2,2]] -> shape (3,)

loss.item()                           # turn a 1-element tensor into a Python float
'''

'sample code\nx = torch.randn(3, 5)                 # 3x5 tensor of random normal numbers\nz = torch.zeros(3, 5)                 # 3x5 tensor of zeros\nt = torch.randint(0, 5, (3,))         # 3 random ints in [0, 5), shape (3,)\n\nx.shape                               # torch.Size([3, 5])\ntorch.exp(x), torch.log(x)            # elementwise\nx.sum(dim=1, keepdim=True)            # sum across each row -> shape (3, 1)\nx / x.sum(dim=1, keepdim=True)        # broadcasting: each row divided by its own sum\nx.mean()                              # average of all elements -> a scalar tensor\n\n# fancy indexing: pick one element per row\nrows = torch.arange(3)                # tensor([0, 1, 2])\ncols = torch.tensor([4, 0, 2])\nx[rows, cols]                         # [x[0,4], x[1,0], x[2,2]] -> shape (3,)\n\nloss.item()                           # turn a 1-element tensor into a Python float\n'

In [2]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

In [3]:
logits = torch.randn(8, 65)
target = torch.randint(0, 65, (8,))

In [4]:
probs = torch.exp(logits) / torch.exp(logits).sum(dim=1, keepdim=True)

In [5]:
rows = torch.arange(target.shape[0])
cols = target
p_correct = probs[rows, cols]
loss = - torch.log(p_correct).mean()

In [6]:
loss

tensor(4.9640)

In [7]:
torch.nn.functional.cross_entropy(logits, target)

tensor(4.9640)

predicted all-zero logits loss: -torch.log(1/65)

In [13]:
all_zero_logits = torch.zeros(8, 65)
F.cross_entropy(all_zero_logits, target)

tensor(4.1744)